In [1]:
!pip install faiss-cpu redis 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 3.7 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [faiss-cpu]/2 [faiss-cpu]


In [2]:
import faiss
import redis
import numpy as np
from sentence_transformers import SentenceTransformer


In [ ]:
# Semantic cache class
class SemanticCache:
    def __init__(self,threshold: float = 0.95, redis_host : str = 'localhost',redis_port: int = 6379):
        self.threshold = threshold
        
        # Encoder 
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        dim = self.encoder.get_embedding_dimension()
        
        self.index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
        
        # K/V store (redis)
        self.redis_client = redis.Redis(host=redis_host,port=redis_port,decode_responses=True)
        self.next_id = 0
    
    def get(self,query:str) -> str | None:
        if self.index.ntotal == 0:
            return None
        
        clean_query = query.strip().lower()
        
        query_vector = self.encoder.encode([clean_query],normalize_embeddings=True)    


        # Search returns distances and integer ids 
        scores , indices = self.index.search(query_vector, k=1)
        
        best_score = scores[0][0]
        best_id = indices[0][0]
        
        print(f"  [DEBUG] Query: '{query}' | Top Score: {best_score:.4f}")
        
        if best_score >= self.threshold:
            return self.redis_client.get(f"cache:{best_id}")
        
        return None
    
    
    def set(self,query:str,answer:str)-> None:
        clean_query = query.strip().lower()
        query_vector = self.encoder.encode([clean_query],normalize_embeddings=True)
        
        # Writee to redis first if this fails , execution halt 
        self.redis_client.set(f"cache:{self.next_id}",answer)
        
        # Write to faiss second if this fails we have a harmless orphand key in redis that will be safely overwitten on the next successful operation 
        vector_id = np.array([self.next_id],dtype=np.int64)
        self.index.add_with_ids(query_vector,vector_id)
        
        self.next_id += 1
        

In [5]:
cache = SemanticCache(threshold=0.95, redis_port=6380)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4127.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
import time
# 1. Populate the cache
query_1 = "What was Apple's total revenue in 2023?"
answer_1 = "Apple's total net sales (revenue) for 2023 was $383.28 billion."
    
cache = SemanticCache(threshold=0.95, redis_port=6380)

print(f"Setting cache for: '{query_1}'")
cache.set(query_1, answer_1)



    # 2. Test an EXACT match (Should Hit)
print("\nTesting Exact Match...")
start = time.time()
result = cache.get("What was Apple's total revenue in 2023?")
print(f"Result: {result}")
print(f"Latency: {(time.time() - start) * 1000:.2f} ms")

    # 3. Test a SEMANTIC match (Different words, same meaning -> Should Hit)
print("\nTesting Semantic Match...")
start = time.time()
result = cache.get("how much revenue did apple make in 2023")
print(f"Result: {result}")
print(f"Latency: {(time.time() - start) * 1000:.2f} ms")

    # 4. Test a MISS (Below threshold -> Should return None)
print("\nTesting Cache Miss (Different metric)...")
start = time.time()
result = cache.get("What was Apple's gross margin in 2023?")
print(f"Result: {result}")
print(f"Latency: {(time.time() - start) * 1000:.2f} ms")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3036.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setting cache for: 'What was Apple's total revenue in 2023?'

Testing Exact Match...
Result: Apple's total net sales (revenue) for 2023 was $383.28 billion.
Latency: 31.35 ms

Testing Semantic Match...
Result: None
Latency: 36.78 ms

Testing Cache Miss (Different metric)...
Result: None
Latency: 62.78 ms
